# Data Generation Master

Combines `generate_demand_capacity_pairs_new_demand_base_cap_pandemic.ipynb` and `pair_dict_generator_new_demand_base_capacity_pandemic.ipynb`. Run all cells top to bottom.

**Outputs (in `data/`):**
1. `production_capacity_scenarios_15YR_applied.csv`: reformatted capacity scenarios
2. `pair_demand_capacity_15YR_NO_MMR_stoch.csv` / `.json`: demand-capacity scenario pairs
3. `scenario_pairs_new_15YR_NO_MMR_stoch.json`: scenario pair dictionary used by the model
4. `scenario_pair_probabilities_new_15YR_NO_MMR_stoch.json`: scenario pair probabilities used by the model

In [ ]:
import sys, os, glob, time, socket
import json
from loguru import logger
from tabulate import tabulate
import pandas as pd
import numpy as np
import joblib
import random

## Settings

In [ ]:
# ANTIGENS = [
#     "Diphtheria",
#     "Pertussis",
#     "Polio",
#     "Tetanus",
#     "Rotavirus",
#     "PCV",
#     "Measles",
#     "Mumps",
#     "Rubella",
#     "Hepatitis_B",
#     "Hib",
#     "HPV",
# ]

ANTIGENS = [
    "Diphtheria",
    "Pertussis",
    "Polio",
    "Tetanus",
    "Rotavirus",
    "PCV",
    "Hepatitis_B",
    "Hib",
    "HPV",
]

N_YEARS = 15
YEARS = np.arange(1, N_YEARS + 1)

In [ ]:
# capacity_scenario_names = [
#     "base_capacity",
#     "pandemic",
# ]
# capacity_scenario_probs = {
#     "base_capacity": 0.8418,
#     "pandemic": 0.0275,
# }

capacity_scenario_names = [
    "base_capacity",
    "IPV_Shortage",
    "pandemic",
    "funding_delay",
    "innacurate_forecast",
    "supply_chain",
    "other",
]
capacity_scenario_probs = {
    "base_capacity": 0.8418,
    "IPV_Shortage": 0.0173,
    "pandemic": 0.0275,
    "funding_delay": 0.0778,
    "innacurate_forecast": 0.0023,
    "supply_chain": 0.0058,
    "other": 0.0275,
}

# capacity_scenario_names = [
#     "base_capacity"
# ]
# capacity_scenario_probs = {
#     "base_capacity": 1
# }

## Step 1: Reformat the capacity scenarios

In [ ]:
dfs = []
for scenario in capacity_scenario_names:
    temp = pd.read_excel(
        f"data/production_capacity_scenarios_15YR_applied.xlsx",
        sheet_name=scenario,
    )
    temp["capacity_scenario"] = scenario
    temp["capacity_scenario_probability"] = capacity_scenario_probs[scenario]
    dfs.append(temp)

print(temp)
# Concatenate all the dataframes
capacity_scenarios = pd.concat(dfs, ignore_index=True)
# rename the columns
capacity_scenarios.rename(columns={"Manufacturer": "manufacturer"}, inplace=True)
# Convert years to columns
capacity_scenarios["capacity"] = capacity_scenarios[YEARS].values.tolist()
# Drop the years columns
capacity_scenarios.drop(YEARS, axis=1, inplace=True)

capacity_scenarios.to_csv("data/production_capacity_scenarios_15YR_applied.csv", index=False)

In [ ]:
capacity_scenarios

## Step 2: Create demand-capacity scenario pairs

In [ ]:
# Load data
demand_scenarios_df = pd.read_csv(
    "data/Medium_Demand_Structured_15YR.csv",
    converters={"demand": pd.eval},
    usecols=["demand", "antigen", "demand_SID", "probability"]
)

demand_scenarios_df2 = demand_scenarios_df[demand_scenarios_df['antigen'].isin(ANTIGENS)]

# capacity_scenarios_df = pd.read_csv(
#     "data/production_capacity_scenarios.csv",
#     converters={"capacity": pd.eval},
# )

capacity_scenarios_df = capacity_scenarios

# Function to generate pairs
def generate_pairs(demand: pd.DataFrame, capacity: pd.DataFrame, n_pairs: int = 15, verbose: bool = True):
    demand_dict = demand.drop_duplicates(subset=["demand_SID"]).set_index("demand_SID")["probability"].to_dict()
    capacity_dict = capacity.drop_duplicates(subset=["capacity_scenario"]).set_index("capacity_scenario")["capacity_scenario_probability"].to_dict()

    demand_keys = list(demand_dict.keys())
    demand_probs = list(demand_dict.values())

    capacity_keys = list(capacity_dict.keys())
    capacity_probs = list(capacity_dict.values())

    if verbose:
        print(f"Unique demand scenarios: {demand_keys}")
        print(f"Unique capacity scenarios: {capacity_keys}")

    pairs = []
    pair_dfs = []
    prob_dict = {}
    pair_idx = 1

    for selected_capacity, selected_prob_capacity in capacity_dict.items():
        available_demand_keys = demand_keys.copy()
        available_demand_probs = demand_probs.copy()

        for _ in range(n_pairs):
            # Normalize the probabilities
            available_demand_probs = [p / sum(available_demand_probs) for p in available_demand_probs]

            selected_demand = np.random.choice(available_demand_keys, p=available_demand_probs)
            # Combine the probabilities
            selected_prob_demand = demand_dict[selected_demand]

            combined_prob = selected_prob_demand * selected_prob_capacity
            prob_dict[pair_idx] = combined_prob

            # Get the selected demand and capacity scenarios
            selected_demand_df = demand[demand["demand_SID"] == selected_demand].copy()
            selected_demand_df["type"] = "antigen"
            selected_demand_df.drop(columns=["probability", "demand_SID"], inplace=True)
            selected_demand_df.rename(columns={"antigen": "unit", "demand": "values"}, inplace=True)

            selected_capacity_df = capacity[capacity["capacity_scenario"] == selected_capacity].copy()
            selected_capacity_df["type"] = "manufacturer"
            selected_capacity_df.drop(columns=["capacity_scenario_probability", "capacity_scenario"], inplace=True)
            selected_capacity_df.rename(columns={"manufacturer": "unit", "capacity": "values"}, inplace=True)

            pair_df = pd.concat([selected_demand_df, selected_capacity_df])

            # Add additional information
            pair_df["pair_idx"] = pair_idx
            pair_df["pair"] = f"Demand: {selected_demand} - Capacity: {selected_capacity}"
            pair_dfs.append(pair_df)
            pairs.append((selected_demand, selected_capacity))
            pair_idx += 1

            # Remove the selected demand from the available demands
            index = available_demand_keys.index(selected_demand)
            available_demand_keys.pop(index)
            available_demand_probs.pop(index)

    pair_df = pd.concat(pair_dfs, ignore_index=True)

    # Calculate the sum of all probabilities
    total_sum = sum(prob_dict.values())

    # Scale the probabilities so they sum up to 1
    scaled_probabilities = {k: v / total_sum for k, v in prob_dict.items()}
    pair_df["pair_probability"] = pair_df["pair_idx"].map(scaled_probabilities)

    return pairs, pair_df

# Generate pairs
scenario_pairs, pair_df = generate_pairs(demand_scenarios_df2, capacity_scenarios_df, n_pairs=1, verbose=True)

# Export to json and csv
pair_df.to_json("data/pair_demand_capacity_15YR_NO_MMR_stoch.json", orient="records", lines=True)
pair_df.to_csv("data/pair_demand_capacity_15YR_NO_MMR_stoch.csv", index=False)

# Display the DataFrame
pair_df

In [ ]:
pair_df['unit'].unique()

In [ ]:
pair_df.groupby('unit').count()

## Step 3: Build the scenario pair dictionary

Reads back the pair CSV written in Step 2, so the JSON is built exactly as before.

In [ ]:
# Load the CSV file
file_path = 'data/pair_demand_capacity_15YR_NO_MMR_stoch.csv'
data = pd.read_csv(file_path)

# Initialize dictionary for scenario pairs
expanded_scenario_pairs = {}

# Iterate over the rows and populate the dictionary
for _, row in data.iterrows():
    pair_idx = row['pair_idx']
    category = 'demand' if row['type'] == 'antigen' else 'capacity'

    # Remove brackets, split space-separated string, and convert to floats
    values_str = row['values'].strip('[]')
    # values = list(map(float, values_str.split()))
    values_str = row['values'].strip('[]').replace(',', '')
    values = list(map(float, values_str.split()))
    unit = row['unit']

    expanded_scenario_pairs.setdefault(pair_idx, {'demand': {}, 'capacity': {}})

    for i, value in enumerate(values, 1):
        expanded_scenario_pairs[pair_idx][category].setdefault(i, {})[unit] = value

expanded_scenario_pairs.keys()

In [ ]:
# Define the file path for the JSON output
json_output_path = 'data/scenario_pairs_new_15YR_NO_MMR_stoch.json'

# Write the dictionary to a JSON file
with open(json_output_path, 'w') as json_file:
    json.dump(expanded_scenario_pairs, json_file)

json_output_path

## Step 4: Save the scenario pair probabilities

In [ ]:
# Load the CSV file into a DataFrame
file_path = 'data/pair_demand_capacity_15YR_NO_MMR_stoch.csv'
df = pd.read_csv(file_path)

# Create the dictionary with scenarios and their probabilities
scenarios_probabilities = df[['pair_idx', 'pair_probability']].drop_duplicates().set_index('pair_idx')['pair_probability'].to_dict()

json_file_path = 'data/scenario_pair_probabilities_new_15YR_NO_MMR_stoch.json'

# Write the dictionary to a JSON file
with open(json_file_path, 'w') as json_file:
    json.dump(scenarios_probabilities, json_file, indent=4)

print(f"Scenarios probabilities have been saved to {json_file_path}")